# Neuronowa sieć konwolucyjna

In [1]:
import numpy as np
from keras.models import Sequential
from keras.layers import *
from keras import optimizers

import matplotlib.pyplot as plt

2025-04-14 11:27:11.660090: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Budowanie modelu

Ostatnio używaliśmy obrazów 32×32-pikselowych - aby ograniczyć liczbę parametrów do uczenia. **W przypadku sieci convnet liczba parametrów nie zależy od rozmiaru obrazu wejściowego**, a jedynie od rozmiaru i liczby filtrów konwolucyjnych. Zatem w przypadku takich sieci można operować na większych obrazach. Tu użyjemy obrazów o rozmiarach 224 × 224 pikseli. Jest to typowy rozmiar wejściowy dla sieci konwolucyjnych (200-300). Można samemu poeksperymentować z większymi obrazami. Niekoniecznie jednak obraz o większej liczbie pikseli da lepsze wyniki rozpoznawania (!)

In [2]:
image_width = 224
image_height = 224
num_classes = 20

Poniżej parametrami Conv2D są kolejno:
- liczba filtrów (np. w pierwszej warstwie konwolucyjnej jest 32, w ostatniej jest 256),
- rozmiar okna konwolucyjnego (filtra), można wpisać (3,3) albo po prostu 3 (co oznacza 3x3),
- w pierwszej warstwie trzeba dodatkowo podać rozmiar wejścia (obrazu na wejściu).

Warstwy MaxPooling2D zmniejszają rozmiar obrazu zmniejszając kosz obliczeń, ale też powodując, że cechy - features będą odpowiadać większym obszarom na oryginalnym obrazie wejściowym.

In [3]:
model = Sequential()
model.add(Conv2D(32, (3,3), padding="same", activation="relu", input_shape=(image_height, image_width, 3)))
model.add(Conv2D(32, 3, padding="same", activation="relu"))
model.add(MaxPooling2D(2))
model.add(Conv2D(64, 3, padding="same", activation="relu"))
model.add(Conv2D(64, 3, padding="same", activation="relu"))
model.add(MaxPooling2D(2))
model.add(Conv2D(128, 3, padding="same", activation="relu"))
model.add(Conv2D(128, 3, padding="same", activation="relu"))
model.add(MaxPooling2D(2))
model.add(Conv2D(256, 3, padding="same", activation="relu"))
model.add(Conv2D(256, 3, padding="same", activation="relu"))
model.add(GlobalAveragePooling2D())
model.add(Dense(num_classes))
model.add(Activation("softmax"))

In [4]:
# Liczba parametrów w pierwszej warstwie to 27*32 + 32 = 896 (bo filtr ma 27 pikseli: 3x3x3 i są 32 filtry + bias)
# W drugiej warstwie konwolucyjnej na wejściu mamy 32 "obrazy" (kanały) 
# a do każdego z nich stosowane są wszystkie filtry (jest ich też 32), 
# zatem liczba parametrów = 3 * 3 * 32 * 32 + 32 = 9248
# bo teraz filtr ma rozmiary 3x3x32 (nie tak jak w pierwszej warstwie 3x3x3).
# W kolejnej warstwie konwolucyjnej są 64 filtry (co można poznać po rozmiarze wyjścia). Na wejściu tej warstwy są 
# 32 kanały ("obrazy"), zatem każdy z 64 filtrów ma wymiary 3x3x32, czyli liczba parametrów 
# jest równa 3*3*32*64 + 64 = 18496 itd. w kolejnych warstwach.
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 224, 224, 32)      896       
                                                                 
 conv2d_1 (Conv2D)           (None, 224, 224, 32)      9248      
                                                                 
 max_pooling2d (MaxPooling2D  (None, 112, 112, 32)     0         
 )                                                               
                                                                 
 conv2d_2 (Conv2D)           (None, 112, 112, 64)      18496     
                                                                 
 conv2d_3 (Conv2D)           (None, 112, 112, 64)      36928     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 56, 56, 64)       0         
 2D)                                                    

Poniższe komórki zawierają kod stosowany w poprzednim przykładzie (regresja logistyczna wykonana bezpośrednio na obrazach o wielkości 32x32 piksele).

In [5]:
model.compile(loss="categorical_crossentropy",
              optimizer=tf.keras.optimizers.Adam(lr=1e-3),
              metrics=["accuracy"])

## Załadownie danych (to samo, co w poprzednim rozważanym przykładzie).

In [6]:
images_dir = "../../DATABASES/snacks/"
train_data_dir = images_dir + "train/"
val_data_dir = images_dir + "val/"
test_data_dir = images_dir + "test/"

In [7]:
def normalize_pixels(image):
    return image / 127.5 - 1

In [8]:
from keras.preprocessing.image import ImageDataGenerator
datagen = ImageDataGenerator(preprocessing_function=normalize_pixels)

In [9]:
batch_size = 64

train_generator = datagen.flow_from_directory(
                    train_data_dir,
                    target_size=(image_width, image_height),
                    batch_size=batch_size,
                    class_mode="categorical",
                    shuffle=True)

val_generator = datagen.flow_from_directory(
                    val_data_dir,
                    target_size=(image_width, image_height),
                    batch_size=batch_size,
                    class_mode="categorical",
                    shuffle=False)

test_generator = datagen.flow_from_directory(
                    test_data_dir,
                    target_size=(image_width, image_height),
                    batch_size=batch_size,
                    class_mode="categorical",
                    shuffle=False)

Found 4838 images belonging to 20 classes.
Found 955 images belonging to 20 classes.
Found 952 images belonging to 20 classes.


In [10]:
index2class = {v:k for k,v in train_generator.class_indices.items()}

## Ocena niewytrenowanego modelu

In [11]:
model.evaluate_generator(test_generator, steps=len(test_generator))

/var/folders/mp/h96dzcrx3rn7d2ljt77qkvlc0000gp/T/ipykernel_7834/633163586.py:1: UserWarning: `Model.evaluate_generator` is deprecated and will be removed in a future version. Please use `Model.evaluate`, which supports generators.
  model.evaluate_generator(test_generator, steps=len(test_generator))
2025-04-14 11:27:14.650707: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]


[2.995042562484741, 0.052521008998155594]

## Training

In [12]:
import warnings
warnings.filterwarnings("ignore")

Metoda fit_generator działająca na modelu zwraca obiekt klasy **History**. zawierający rózne metryki (loss - wartość funcji kosztu, val_loss - wartość funkcji kosztu dla validation set, acc - accuracy, val_acc - validation accuracy). 

Utwórzmy listę takich obiektów (tablicę):

In [13]:
histories = []

In [ ]:
history = model.fit_generator(train_generator, 
                              steps_per_epoch=len(train_generator),
                              validation_data=val_generator,
                              validation_steps=len(val_generator),
                              epochs=5,
                              workers=8)
# Dodajemy obiekt history do listy histories:
histories.append(history)
#
# ETA - "Estimated Time of Arrival" - przybliżony czas zakończenia obliczeń dla epoki.

Epoch 1/5


2025-04-14 11:27:30.177765: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]


22/76 [=======>......................] - ETA: 4:16 - loss: 2.9948 - accuracy: 0.0568

In [ ]:
history2 = model.fit_generator(train_generator, 
                                     steps_per_epoch=len(train_generator),
                                     validation_data=val_generator,
                                     validation_steps=len(val_generator),
                                     epochs=25,
                                     workers=8)
                 
histories.append(history2)

In [ ]:
def combine_histories():
    history = { "loss": [], "val_loss": [], "accuracy": [], "val_accuracy": [] }
    for h in histories:
        for k in history.keys():
            history[k] += h.history[k]
    return history

In [ ]:
history = combine_histories()

In [ ]:
def plot_loss(history):
    fig = plt.figure(figsize=(10, 6))
    plt.plot(history["loss"])
    plt.plot(history["val_loss"])
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(["Train", "Validation"])
    plt.show()

In [ ]:
plot_loss(history)

In [ ]:
def plot_accuracy(history):
    fig = plt.figure(figsize=(10, 6))
    plt.plot(history["accuracy"])
    plt.plot(history["val_accuracy"])
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(["Train", "Validation"])
    plt.show()

In [ ]:
plot_accuracy(history)

## Zmniejszenie współczynnika uczenia (learning rate)

In [ ]:
import keras.backend as K
K.set_value(model.optimizer.lr, K.get_value(model.optimizer.lr) / 10)
#
# Zatem po 30 epokach współczynnik uczenia został zmniejszony 10 razy.

In [ ]:
histories.append(model.fit_generator(train_generator, 
                                     steps_per_epoch=len(train_generator),
                                     validation_data=val_generator,
                                     validation_steps=len(val_generator),
                                     epochs=10,
                                     workers=8))

In [ ]:
history = combine_histories()

In [ ]:
plot_loss(history)

In [ ]:
plot_accuracy(history)

## Sprawdzenie jakości modelu na zbiorze testowym

In [ ]:
model.evaluate_generator(test_generator, steps=len(test_generator))

Ostatecznie otrzymaliśmy dokładność (accuracy) rzędu **40%, czyli znacznie lepiej niż w przypadku zastosowania regresji logistycznej bezpośrednio do (spłaszczonego) obrazu**. 

**Jednak nie jest to jeszcze wielkość zadowalająca**.